In [2]:
import os
import psutil
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [11]:
# STEP 1. Model load(lightweight model)
model_id = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 2793.60it/s, Materializing param=model.norm.weight]                                


In [24]:
def run_inference(prompt="Explain the core of a CPU in one sentence."):
    inputs = tokenizer(prompt, return_tensors="pt")
    start_time = time.time()

    # Run CPU inference
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=20)

    end_time = time.time()
    return end_time - start_time

In [22]:
def set_cpu_affinity(cpu_ids):
    # Fix CPU core which current process will use.
    p = psutil.Process(os.getpid())
    p.cpu_affinity(cpu_ids)
    print(f"Currently set CPU core: {p.cpu_affinity()}")

In [30]:
# --- Start experiment ---
prompt_text = "What is the difference between P-core and E-core?"

# Experiment 1: Only use P-core(0~3 threads usage)
print("\nExperiment 1. Inferring P-core usage.")
set_cpu_affinity([0,1,2,3])
torch.set_num_threads(4)
p_time = run_inference(prompt_text)
print(f"P-core inference time: {p_time:.2f} sec.")

# Experiment 2: Only use E-core(16~19 threads usage)
print("\nExperiment 2. Inferring E-core usage.")
set_cpu_affinity([16,17,18,19])
torch.set_num_threads(4)
e_time = run_inference(prompt_text)
print(f"E-core inference time: {e_time:.2f} sec.")

print(f"\nPerformance differences: E-core is slower about {e_time/p_time:.1f} than P-core.")

# Experiment 3: Mixed (P-core and E-core)
print("\nExperiment 3. Inferring Mix-core usage.")
set_cpu_affinity([0,1,16,17])
torch.set_num_threads(4)
m_time = run_inference(prompt_text)
print(f"Mix-core inference time: {m_time:.2f} sec.")

# Experiment 4: Random 
import random
print("\nExperiment 4. Inffering Random-core usage.")
all_cores = list(range(20))
random_cores = random.sample(all_cores, 4)
set_cpu_affinity(random_cores)
torch.set_num_threads(4)
r_time = run_inference(prompt_text)
print(f"Random-core inference time: {r_time:.2f} sec.")


Experiment 1. Inferring P-core usage.
Currently set CPU core: [0, 1, 2, 3]
P-core inference time: 2.64 sec.

Experiment 2. Inferring E-core usage.
Currently set CPU core: [16, 17, 18, 19]
E-core inference time: 3.71 sec.

Performance differences: E-core is slower about 1.4 than P-core.

Experiment 3. Inferring Mix-core usage.
Currently set CPU core: [0, 1, 16, 17]
Mix-core inference time: 3.71 sec.

Experiment 4. Inffering Random-core usage.
Currently set CPU core: [0, 4, 6, 13]
Random-core inference time: 2.64 sec.


In [3]:
import multiprocessing as mp
import os
import psutil
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# CPU Optimal Scheduling Prac.
import multiprocessing as mp

# --- Define independent Worker process. ---
def model_worker(worker_name, cpu_ids, input_queue, output_queue):
    p = psutil.Process(os.getpid())
    p.cpu_affinity(cpu_ids)

    torch.set_num_threads(len(cpu_ids))

    print(f"{worker_name} Start. (PID: {os.getpid()} -> Core {cpu_ids}")
    model_id = "google/gemma-3-1b-it"
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(model_id)
    except Exception as e:
        output_queue.put({"error" : str(e)})
        return
    
    print(f"{worker_name} model loading completed. Waiting..")

    while True:
        task = input_queue.get()
        if task is None:
            break
        task_id, prompt, priority = task
        start_t = time.time()

        # Start inference.
        inputs = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=30)

        duration = time.time() - start_t

        output_queue.put({
            "task_id":task_id,
            "worker":worker_name,
            "priority":priority,
            "duration":duration
        })

In [4]:
# --- Schduler(Main process)---
def run_optimized_scheduler():
    # Queue generate.
    high_priority_queue = mp.Queue() # P-core
    low_priority_queue = mp.Queue() # E-core
    result_queue = mp.Queue()

    # Scheduling strategy
    # P-Worker: Number 0~3 Core (Fast) -> High Queue
    p_worker = mp.Process(
        target=model_worker,
        args=("Fast Worker", [0,1,2,3], high_priority_queue, result_queue)
    )
    # E-worker: Number 16~19 Core (Slow) -> Low Queue
    e_worker = mp.Process(
        target=model_worker,
        args=("Slow Worker", [16,17,18,19], low_priority_queue, result_queue)
    )
    print("--- [Scheudler] Worker process start ---")
    p_worker.start()
    e_worker.start()

    time.sleep(15)

    # Task scenario (Important task 3, Common task 3)
    tasks = [
        (1, "Important: Large language model summary", "HIGH"),
        (2, "Background: Write a poem about a cat.", "LOW"),
        (3, "Important: Explain Multiprocessing", "HIGH"),
        (4, "Background: Count to 100", "LOW"),
        (5, "Important: What is Cloud Computing?", "HIGH"),
        (6, "Background: How was your feeling?", "LOW")
    ]
    print("\n ---[Scheduler] Task routing start ---")
    start_global = time.time()

    for t_id, prompt,prio in tasks:
        if prio == "HIGH":
            print(f"Assigning Task {t_id} (HIGH) -> Fast Lane")
            high_priority_queue.put((t_id, prompt, prio))
        else:
            print(f"Assigning Task {t_id} (LOW) -> Slow Lane")
            low_priority_queue.put((t_id, prompt, prio))
    
    high_priority_queue.put(None)
    low_priority_queue.put(None)

    completed = 0
    while completed < len(tasks):
        res = result_queue.get()
        print(f"Task {res['task_id']} ({res['priority']}) completed by {res['worker']} | Duration: {res['duration']:.2f} sec.")
        completed += 1

    total_time = time.time() - start_global
    print(f"\n Totally completed time: {total_time:.2f} sec.")

    p_worker.join()
    e_worker.join()

if __name__ == "__main__":
    try:
        mp.set_start_method('fork', force=True)
    except RuntimeError:
        pass
    run_optimized_scheduler()

--- [Scheudler] Worker process start ---
Fast Worker Start. (PID: 34187 -> Core [0, 1, 2, 3]
Slow Worker Start. (PID: 34193 -> Core [16, 17, 18, 19]


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Fast Worker model loading completed. Waiting..


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Slow Worker model loading completed. Waiting..

 ---[Scheduler] Task routing start ---
Assigning Task 1 (HIGH) -> Fast Lane
Assigning Task 2 (LOW) -> Slow Lane
Assigning Task 3 (HIGH) -> Fast Lane
Assigning Task 4 (LOW) -> Slow Lane
Assigning Task 5 (HIGH) -> Fast Lane
Assigning Task 6 (LOW) -> Slow Lane
Task 1 (HIGH) completed by Fast Worker | Duration: 3.22 sec.
Task 2 (LOW) completed by Slow Worker | Duration: 4.88 sec.
Task 3 (HIGH) completed by Fast Worker | Duration: 3.16 sec.
Task 4 (LOW) completed by Slow Worker | Duration: 4.81 sec.
Task 5 (HIGH) completed by Fast Worker | Duration: 3.30 sec.
Task 6 (LOW) completed by Slow Worker | Duration: 4.20 sec.

 Totally completed time: 13.88 sec.
